In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, accuracy_score
import joblib

def voto_mayoria(grupo):
    return grupo.value_counts().idxmax()

In [3]:
df_adultos = pd.read_excel('dataset_physionet2016.xlsx')
df_adultos['paciente_id'] = df_adultos['paciente_id'].astype(str)

feature_cols_adultos = [f"MFCC_{i+1}" for i in range(13)] + ["RMS"]
X_adultos = df_adultos[feature_cols_adultos].values
y_adultos = df_adultos['Etiqueta'].values
grupos_adultos = df_adultos['paciente_id'].values

sgkf_split_adultos = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
idx_dev, idx_prueba_final = next(sgkf_split_adultos.split(X_adultos, y_adultos, groups=grupos_adultos))

X_dev_ad, y_dev_ad, grupos_dev_ad = X_adultos[idx_dev], y_adultos[idx_dev], grupos_adultos[idx_dev]
X_prueba_ad, y_prueba_ad, grupos_prueba_ad = X_adultos[idx_prueba_final], y_adultos[idx_prueba_final], grupos_adultos[idx_prueba_final]

pacientes_dev_ad = set(grupos_dev_ad)
pacientes_prueba_ad = set(grupos_prueba_ad)
print(f"pacientes repetidos entre dev y prueba: {len(pacientes_dev_ad & pacientes_prueba_ad)}")
print(f"dev: {len(idx_dev)} filas, {len(pacientes_dev_ad)} sujetos")
print(f"prueba final: {len(idx_prueba_final)} filas, {len(pacientes_prueba_ad)} sujetos")

pd.Series(sorted(pacientes_prueba_ad)).to_csv('sujetos_prueba_final_adultos.csv', index=False, header=['paciente_id'])
print("guardado: sujetos_prueba_final_adultos.csv")

pacientes repetidos entre dev y prueba: 0
dev: 20138 filas, 2261 sujetos
prueba final: 4993 filas, 564 sujetos
guardado: sujetos_prueba_final_adultos.csv


In [9]:
sgkf_adultos = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0),
    'Gradient Boosting': GradientBoostingClassifier(random_state=0),
}

for nombre, modelo in modelos.items():
    pipe = Pipeline([('escalador', StandardScaler()), ('clf', modelo)])
    scores = cross_val_score(pipe, X_dev_ad, y_dev_ad, cv=sgkf_adultos, groups=grupos_dev_ad, scoring='accuracy')
    print(f"{nombre:22s} exactitud = {scores.mean():.3f} +/- {scores.std():.3f}")

Regresion Logistica    exactitud = 0.828 +/- 0.016
SVM                    exactitud = 0.896 +/- 0.004
Random Forest          exactitud = 0.915 +/- 0.010
Gradient Boosting      exactitud = 0.913 +/- 0.010


In [ ]:
mejor_pipe_ad = Pipeline([('escalador', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0))])


np.save('proba_dev_adultos.npy', proba_dev_ad)
print("listo y guardado")

In [13]:
proba_dev_ad = cross_val_predict(mejor_pipe_ad, X_dev_ad, y_dev_ad, cv=sgkf_adultos, groups=grupos_dev_ad, method='predict_proba')
df_dev_ad = df_adultos.iloc[idx_dev].copy()
df_dev_ad['prob_soplo'] = proba_dev_ad[:, 1]
etiquetas_dev_paciente_ad = df_dev_ad.groupby('paciente_id')['Etiqueta'].first()

print(f"{'umbral':>8} {'sensibilidad':>13} {'especificidad':>15} {'Youden J':>10}")
mejor_j_ad, mejor_umbral_ad = -1, None
for umbral in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]:
    df_dev_ad['pred_ciclo'] = (df_dev_ad['prob_soplo'] >= umbral).astype(int) * 2
    pred_paciente = df_dev_ad.groupby('paciente_id')['pred_ciclo'].apply(voto_mayoria)
    m = confusion_matrix(etiquetas_dev_paciente_ad, pred_paciente)
    tn, fp, fn, vp = m.ravel()
    sens, esp = vp/(vp+fn), tn/(tn+fp)
    j = sens + esp - 1
    print(f"{umbral:>8.2f} {sens:>13.3f} {esp:>15.3f} {j:>10.3f}")
    if j > mejor_j_ad:
        mejor_j_ad, mejor_umbral_ad = j, umbral

print(f"\numbral elegido: {mejor_umbral_ad}")

  umbral  sensibilidad   especificidad   Youden J
    0.50         0.573           0.978      0.551
    0.40         0.665           0.965      0.630
    0.30         0.777           0.937      0.714
    0.25         0.837           0.913      0.750
    0.20         0.910           0.885      0.795
    0.15         0.949           0.831      0.780
    0.10         0.981           0.768      0.749

umbral elegido: 0.2


In [4]:
X_prueba_ad_arr = df_adultos[df_adultos['paciente_id'].isin(pacientes_prueba_ad)][feature_cols_adultos].values
y_prueba_ad_arr = df_adultos[df_adultos['paciente_id'].isin(pacientes_prueba_ad)]['Etiqueta'].values

modelo_dev_ad = Pipeline([('escalador', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0))])
modelo_dev_ad.fit(X_dev_ad, y_dev_ad)

proba_final_ad = modelo_dev_ad.predict_proba(X_prueba_ad_arr)[:, 1]
np.save('proba_final_adultos.npy', proba_final_ad)
print("listo y guardado")

listo y guardado


In [14]:
df_prueba_ad = df_adultos[df_adultos['paciente_id'].isin(pacientes_prueba_ad)].copy()
df_prueba_ad['prob_soplo'] = proba_final_ad
df_prueba_ad['pred_ciclo'] = (df_prueba_ad['prob_soplo'] >= mejor_umbral_ad).astype(int) * 2

pred_paciente_final_ad = df_prueba_ad.groupby('paciente_id')['pred_ciclo'].apply(voto_mayoria)
etiquetas_paciente_final_ad = df_prueba_ad.groupby('paciente_id')['Etiqueta'].first()

m_final_ad = confusion_matrix(etiquetas_paciente_final_ad, pred_paciente_final_ad)
tn, fp, fn, vp = m_final_ad.ravel()
sens_final_ad, esp_final_ad = vp/(vp+fn), tn/(tn+fp)

auc_final_ad = roc_auc_score(y_prueba_ad_arr, proba_final_ad)

print(f"EXAMEN FINAL (adultos), con {len(etiquetas_paciente_final_ad)} sujetos nunca vistos:")
print(f"Sensibilidad: {sens_final_ad:.3f}   Especificidad: {esp_final_ad:.3f}")
print(f"ROC-AUC (por ciclo): {auc_final_ad:.3f}")
print(m_final_ad)

EXAMEN FINAL (adultos), con 564 sujetos nunca vistos:
Sensibilidad: 0.899   Especificidad: 0.881
ROC-AUC (por ciclo): 0.957
[[401  54]
 [ 11  98]]


In [15]:
import os

BASE_DIR = r"C:\Users\emigo\OneDrive\Documentos\Servicio Social\classification-of-heart-sound-recordings\classification-of-heart-sound-recordings-the-physionet-computing-in-cardiology-challenge-2016-1.0.0"

sqi_frames = []
for carpeta in ['training-a', 'training-b', 'training-c', 'training-d', 'training-e', 'training-f']:
    ruta_sqi = os.path.join(BASE_DIR, carpeta, 'REFERENCE-SQI.csv')
    if os.path.exists(ruta_sqi):
        sqi = pd.read_csv(ruta_sqi, header=None, names=['nombre', 'label', 'sqi'])
        sqi['archivo'] = carpeta + '_' + sqi['nombre']
        sqi_frames.append(sqi[['archivo', 'sqi']])
        print(f"{carpeta}: SQI disponible, {len(sqi)} registros")
    else:
        print(f"{carpeta}: SQI no disponible")

df_sqi = pd.concat(sqi_frames, ignore_index=True)
print(f"\ntotal con dato de calidad: {len(df_sqi)}")

# entre los sujetos de PRUEBA (nunca vistos) que de verdad son Sano,
# comparamos la probabilidad promedio de "Soplo" segun la calidad de su grabacion
resumen_prueba = df_prueba_ad.groupby('archivo').agg(real=('Etiqueta', 'first'), prob_promedio=('prob_soplo', 'mean')).reset_index()
resumen_prueba = resumen_prueba.merge(df_sqi, on='archivo', how='inner')

sanos = resumen_prueba[resumen_prueba['real'] == 0]
print(f"\nsujetos realmente Sanos con dato de calidad: {len(sanos)}")
print(sanos.groupby('sqi')['prob_promedio'].describe())

training-a: SQI disponible, 409 registros
training-b: SQI disponible, 490 registros
training-c: SQI disponible, 31 registros
training-d: SQI disponible, 55 registros
training-e: SQI disponible, 2141 registros
training-f: SQI disponible, 114 registros

total con dato de calidad: 3240

sujetos realmente Sanos con dato de calidad: 455
     count      mean       std  min     25%       50%       75%     max
sqi                                                                    
0     40.0  0.101154  0.119409  0.0  0.0055  0.029000  0.164062  0.4000
1    415.0  0.067278  0.118919  0.0  0.0050  0.013333  0.066071  0.7135


In [16]:
sqi_frames = []
for carpeta in ['training-a', 'training-b', 'training-c', 'training-d', 'training-e', 'training-f']:
    ruta_sqi = os.path.join(BASE_DIR, carpeta, 'REFERENCE-SQI.csv')
    sqi = pd.read_csv(ruta_sqi, header=None, names=['nombre', 'label', 'sqi'])
    sqi['archivo'] = carpeta + '_' + sqi['nombre']
    sqi_frames.append(sqi[['archivo', 'sqi']])
df_sqi_completo = pd.concat(sqi_frames, ignore_index=True)

df_adultos_v2 = df_adultos.merge(df_sqi_completo, on='archivo', how='left')
n_antes = df_adultos_v2['paciente_id'].nunique()
df_adultos_v2 = df_adultos_v2[df_adultos_v2['sqi'] == 1]
print(f"solo buena calidad: {df_adultos_v2['paciente_id'].nunique()} de {n_antes} sujetos")

solo buena calidad: 2533 de 2825 sujetos
